In [1]:
%pip install pypdf langchain-community langchain-text-splitters chromadb langchain-google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = r"C:\Users\PC\Downloads\karthika journal.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Pages loaded:", len(pages))

C:\Users\PC\AppData\Local\Temp\ipykernel_20628\3840211371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 5


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print("Number of chunks:", len(chunks))

Number of chunks: 48


In [4]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Embedding model ready!")

Embedding model ready!


In [5]:
chunk_texts = [chunk.page_content for chunk in chunks]

chunk_vectors = embeddings.embed_documents(chunk_texts)

print("Number of vectors:", len(chunk_vectors))
print("Vector dimensions:", len(chunk_vectors[0]))

Number of vectors: 48
Vector dimensions: 3072


In [6]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.create_collection(
    name="vehicle_damage_docs"
)

print("Chroma database and collection created!")

Chroma database and collection created!


In [7]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunk_texts,
    embeddings=chunk_vectors
)

print("48 chunks stored in Chroma!")

48 chunks stored in Chroma!


In [8]:
query = "What are the three severity levels?"

query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

print("Retrieved chunks:\n")

for i, doc in enumerate(results["documents"][0]):
    print(f"--- Chunk {i + 1} ---")
    print(doc)
    print()

Retrieved chunks:

--- Chunk 1 ---
moderate, indicating more significant damage covering a larger 
surface area or affecting panel alignment; and severe, 
representing extensive structural damage requiring major repair 
intervention. The severity assignment assists users and 
insurance assessors in making informed decisions regarding 
repair prioritization and claim processing.   
E.  Database and Output Module  
Detection results including bounding box coordinates, 
damage categories, confidence scores , and severity

--- Chunk 2 ---
separation or missing vehicle parts. 
D.  Severity Classification Module  
Following damage detection, the severity classification 
module analyses the spatial extent, depth characteristics, and 
distribution of each identified damage region to assign a 
severity rating. Damage instances are categorized into three 
levels: minor, referring to superficial surface marks with 
limited extent th at do not affect vehicle structural integrity;

--- Chunk 3 ---


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)

print("Gemini chat model ready!")

Gemini chat model ready!


In [10]:
context = "\n\n".join(results["documents"][0])

prompt = f"""
Answer the question using only the context provided below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Answer clearly and concisely.
"""

print(prompt)


Answer the question using only the context provided below.

Context:
moderate, indicating more significant damage covering a larger 
surface area or affecting panel alignment; and severe, 
representing extensive structural damage requiring major repair 
intervention. The severity assignment assists users and 
insurance assessors in making informed decisions regarding 
repair prioritization and claim processing.   
E.  Database and Output Module  
Detection results including bounding box coordinates, 
damage categories, confidence scores , and severity

separation or missing vehicle parts. 
D.  Severity Classification Module  
Following damage detection, the severity classification 
module analyses the spatial extent, depth characteristics, and 
distribution of each identified damage region to assign a 
severity rating. Damage instances are categorized into three 
levels: minor, referring to superficial surface marks with 
limited extent th at do not affect vehicle structural integrity

In [11]:
response = llm.invoke(prompt)

print("Answer:")
print(response.content)

C:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer:
[{'type': 'text', 'text': 'Based on the provided context, the three severity levels are:\n\n1. **Minor**\n2. **Moderate**\n3. **Severe**', 'extras': {'signature': 'EuMICuAIARFNMg8mHRSgy/lWXRyWksCmeGsbA98G1oClVjhd841JQ5+TduOV/li5+vFgn1H1LrErcFBSpf2uiLuVtFixeLVmv45wE/YbntCQjNgUIlgiJ2sr7KXQEwcvK7iL0Gm+9Khnhz/KPiClJDKqpPQs4mMkULX0xJha3myauWtBdZFP6nHifNUfblnG7MYU7qRXRaEh6RFi6uuSU2PV0SArb5Yr1BmitdHE23zeF0wWYJP5BNb4p1h1Ez6OPyU7iuC5/KlRmzHVV0ge6jhmQsaX68zD7TFDsb47G0qeVuVY+jSN1S3N8z/ls29Ypy47YccHk/S2uggLMESwyWiwUMPNamaJdJ6ycKEx6+FNZJzMzATUFYYC1p1KW9i7DqEI1soLpCCItvLfrIgXqE/sdvyFWyJ3g6qwv4cBD+vvODvfWoOz96qkidxYZnmEBnJpt9JGdsQL38de48tMjgbnRydMlahUEaRo871KwWzl5Am37YYMnl2M9dKiYdAZOTzkBJ3z+8Q1o6qUG1BSDKlfcum+KNg2E65Azow+d1BZ1j5Fp8YkoHwGW2Zw6nqcmJctX2Uj9xUlCiemKdJGbFWTdZiKBcXAOFP3lSQINwp+gGE3tZz2ahAyPcCDJvIE+cBpefTBNXzyi3T5MHps/UEYYS96SJW1XUjDC7S9f017Z1BF24x7nigdg1Sa2Rejh7tAKcumDTYEh1BJeyqRe2//2SQbDEzoWjQvj+ftVGFcJjI9e2nf45Gcn+zIyVxEyRo660kMsuyqBLJw7Ekamj+ncfbvonsIlm8aC30T4Bbo+d7Xt7xoqKUtgiCM

In [12]:
query = "What are the four primary vehicle damage categories?"

query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

context = "\n\n".join(results["documents"][0])

prompt = f"""
Answer the question using only the context provided below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Answer clearly and concisely.
"""

response = llm.invoke(prompt)

print("Answer:")
print(response.content)

C:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer:
[{'type': 'text', 'text': 'Based on the provided context, the four primary vehicle damage categories are:\n\n1. **Scratches**\n2. **Dents**\n3. **Cracks**\n4. **Broken components** (or broken vehicle components)', 'extras': {'signature': 'Et4LCtsLARFNMg+YkFt6xZxA8/aI1l36TUNWYAF6K72Ion9jlim+f4SvJjuVTDF44DmHy+vohLQYnIg4TSfTh6KniSogH4261RJVQkOhG4DI5ytpsmXc0US0VPQhBtCAzcfWppzwQqqjjGrXnRSdHiRvJNaH8vuDLWasldtUlU4qDr+I5OwcKZ94cNnMFUDEWHyEeo44ccAVqFcOps1atrJV33ysnRfkJ645VE+tmdQdlclFf/ZbiNeVNKLhrAWvfQILus6U9G05rOdjS6FUPGkZT+dBTOwa5jMMMnwlcdV8yAYRSTbQ/yC1KIIRk2fAVkvLGCCctzCPub6iZUIIkkfZFpzDChIbNQclEz3jr6T7rG9XtdXQ1zwn3r2GUrr2VvF1pRjy9I2mKyFvVi+f8wmwRBj1DxZi1aBG+zCPaEv5K+nNqMT7YCJOycIH14QFD4oR8oUfRvxqD6zx+qZazhkOKMIVjAV6oCnbcc+wtJX2he18GWoCyDo88woBtIWEwk7FZ15ak4hDzca4+Gi55ljvauT+FFhcDzUkHV1QEMlNlwU4aAs88giopo2zWL/EiF/LrGN5/VAfNAJaJFefjnUdshmFzqOglxTwvQguw/fx/NlmNVOW3NVgikyyW8xMaRXjd+G/AYrgVxPgUV1r9ZJVSrfmkC3AVMvJxvpnfYVdMj/19OOnBrT6FY9jOOnYn03CLOA4nwVMHM7NJGmplLUHJeim5BOPxe5putWW+SoJ8EsAn

In [13]:
print("RAW RESPONSE:")
print(response)

RAW RESPONSE:
content=[{'type': 'text', 'text': 'Based on the provided context, the four primary vehicle damage categories are:\n\n1. **Scratches**\n2. **Dents**\n3. **Cracks**\n4. **Broken components** (or broken vehicle components)', 'extras': {'signature': 'Et4LCtsLARFNMg+YkFt6xZxA8/aI1l36TUNWYAF6K72Ion9jlim+f4SvJjuVTDF44DmHy+vohLQYnIg4TSfTh6KniSogH4261RJVQkOhG4DI5ytpsmXc0US0VPQhBtCAzcfWppzwQqqjjGrXnRSdHiRvJNaH8vuDLWasldtUlU4qDr+I5OwcKZ94cNnMFUDEWHyEeo44ccAVqFcOps1atrJV33ysnRfkJ645VE+tmdQdlclFf/ZbiNeVNKLhrAWvfQILus6U9G05rOdjS6FUPGkZT+dBTOwa5jMMMnwlcdV8yAYRSTbQ/yC1KIIRk2fAVkvLGCCctzCPub6iZUIIkkfZFpzDChIbNQclEz3jr6T7rG9XtdXQ1zwn3r2GUrr2VvF1pRjy9I2mKyFvVi+f8wmwRBj1DxZi1aBG+zCPaEv5K+nNqMT7YCJOycIH14QFD4oR8oUfRvxqD6zx+qZazhkOKMIVjAV6oCnbcc+wtJX2he18GWoCyDo88woBtIWEwk7FZ15ak4hDzca4+Gi55ljvauT+FFhcDzUkHV1QEMlNlwU4aAs88giopo2zWL/EiF/LrGN5/VAfNAJaJFefjnUdshmFzqOglxTwvQguw/fx/NlmNVOW3NVgikyyW8xMaRXjd+G/AYrgVxPgUV1r9ZJVSrfmkC3AVMvJxvpnfYVdMj/19OOnBrT6FY9jOOnYn03CLOA4nwVMHM7NJGmplLUHJeim5BOPxe5

In [14]:
print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, the four primary vehicle damage categories are:\n\n1. **Scratches**\n2. **Dents**\n3. **Cracks**\n4. **Broken components** (or broken vehicle components)', 'extras': {'signature': 'Et4LCtsLARFNMg+YkFt6xZxA8/aI1l36TUNWYAF6K72Ion9jlim+f4SvJjuVTDF44DmHy+vohLQYnIg4TSfTh6KniSogH4261RJVQkOhG4DI5ytpsmXc0US0VPQhBtCAzcfWppzwQqqjjGrXnRSdHiRvJNaH8vuDLWasldtUlU4qDr+I5OwcKZ94cNnMFUDEWHyEeo44ccAVqFcOps1atrJV33ysnRfkJ645VE+tmdQdlclFf/ZbiNeVNKLhrAWvfQILus6U9G05rOdjS6FUPGkZT+dBTOwa5jMMMnwlcdV8yAYRSTbQ/yC1KIIRk2fAVkvLGCCctzCPub6iZUIIkkfZFpzDChIbNQclEz3jr6T7rG9XtdXQ1zwn3r2GUrr2VvF1pRjy9I2mKyFvVi+f8wmwRBj1DxZi1aBG+zCPaEv5K+nNqMT7YCJOycIH14QFD4oR8oUfRvxqD6zx+qZazhkOKMIVjAV6oCnbcc+wtJX2he18GWoCyDo88woBtIWEwk7FZ15ak4hDzca4+Gi55ljvauT+FFhcDzUkHV1QEMlNlwU4aAs88giopo2zWL/EiF/LrGN5/VAfNAJaJFefjnUdshmFzqOglxTwvQguw/fx/NlmNVOW3NVgikyyW8xMaRXjd+G/AYrgVxPgUV1r9ZJVSrfmkC3AVMvJxvpnfYVdMj/19OOnBrT6FY9jOOnYn03CLOA4nwVMHM7NJGmplLUHJeim5BOPxe5putWW+SoJ8EsAnxkjhXJle

In [15]:
print(response.content[0]["text"])

Based on the provided context, the four primary vehicle damage categories are:

1. **Scratches**
2. **Dents**
3. **Cracks**
4. **Broken components** (or broken vehicle components)


# Conclusion

Today I built a basic RAG (Retrieval-Augmented Generation) pipeline using a PDF document, Gemini embeddings, Chroma, and Gemini.

The pipeline extracts text from a PDF, splits it into chunks, converts the chunks into embeddings, stores them in Chroma, retrieves relevant chunks for a question, and passes the retrieved context to Gemini to generate an answer.

I tested the pipeline with questions about the vehicle damage categories and severity levels, and it successfully retrieved the relevant information from the document.

### RAG Pipeline

PDF → Chunks → Embeddings → Chroma → Retrieval → Gemini → Answer

### Key Takeaways

- RAG allows an LLM to answer questions using information from a specific document.
- Embeddings represent text as numerical vectors.
- Chroma can store and retrieve vector embeddings.
- Retrieval provides relevant context to the LLM.
- The LLM generates the final answer using the retrieved context.

**Day 18 Complete! 🚀**

In [16]:
readme = """# Day 18 - Basic RAG Pipeline

## Objective

Build a basic Retrieval-Augmented Generation (RAG) pipeline using a PDF document, Gemini embeddings, Chroma, and Gemini.

## What I Built

The pipeline follows this flow:

PDF → Chunks → Embeddings → Chroma → Retrieval → Gemini → Answer

## Steps

1. Loaded a PDF using `PyPDFLoader`.
2. Split the document into 48 chunks using `RecursiveCharacterTextSplitter`.
3. Generated 3072-dimensional embeddings using Gemini.
4. Stored the chunks and embeddings in a persistent Chroma vector database.
5. Converted user questions into embeddings.
6. Retrieved the most relevant chunks from Chroma.
7. Passed the retrieved context to Gemini.
8. Generated answers using only the retrieved document context.

## Technologies Used

- Python
- LangChain
- Gemini
- Gemini Embeddings
- Chroma
- PyPDF

## Example Questions

### Question 1
What are the three severity levels?

**Answer:** Minor, Moderate, Severe.

### Question 2
What are the four primary vehicle damage categories?

**Answer:** Scratches, Dents, Cracks, and Broken components.

## Key Takeaways

- RAG allows an LLM to answer questions using information from a specific document.
- Embeddings represent text as numerical vectors.
- Chroma stores and retrieves vector embeddings.
- Retrieval provides relevant context to the LLM.
- The LLM generates the final answer using the retrieved context.

## Conclusion

Today I built a complete basic RAG pipeline that connects document retrieval with Gemini generation. This is the foundation for building document question-answering systems.

**Day 18 Complete! 🚀**
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created successfully!")

README.md created successfully!
